# Representation Tradeoffs and Verification

```{contents}
:local:
:depth: 2
```

This lab compares array-backed and linked-node representations by counting visits and explaining tradeoffs. The goal is not to prove that one representation is always better. The goal is to connect a workload to a representation choice.


```{index} Array Search and Indexing
```



```{index} Array Search and Indexing
```

## Array Search and Indexing



Arrays are strong at indexing. Searching by value still requires checking elements until the value is found or the array ends.

The example counts visits so the two costs can be compared: a search by value, which must inspect elements until it finds the target, and a read by index, which needs no visits at all.


In [ ]:
int[] values = { 12, 7, 19, 4, 15 };

int visits = 0;
int target = 15;
for (int index = 0; index < values.Length; index++)
{
    visits++;
    if (values[index] == target)
    {
        Console.WriteLine($"Found {target} at index {index} after {visits} visits.");
        break;
    }
}

Console.WriteLine($"Direct index values[2] = {values[2]}");


Found 15 at index 4 after 5 visits.
Direct index values[2] = 19


The output is `Found 15 at index 4 after 5 visits.` and `Direct index values[2] = 19`. The target sits in the last slot, so the scan inspected every element; a missing target would also cost five visits. The direct read is one arithmetic step from the index to the slot, no matter how long the array is. That asymmetry is what the linked version below cannot offer.


```{index} Linked Traversal
```



```{index} Linked Traversal
```

## Linked Traversal



A linked representation visits nodes by following references. There is no direct index jump.

The same five values and the same target, stored as a chain of nodes. The visit counter counts how many `Next` references the loop followed before it found the value.


In [ ]:
Node<int> head = new Node<int>(12);
head.Next = new Node<int>(7);
head.Next.Next = new Node<int>(19);
head.Next.Next.Next = new Node<int>(4);
head.Next.Next.Next.Next = new Node<int>(15);

int visits = 0;
int target = 15;
for (Node<int>? current = head; current is not null; current = current.Next)
{
    visits++;
    if (current.Value == target)
    {
        Console.WriteLine($"Found {target} after {visits} node visits.");
        break;
    }
}

public class Node<T>
{
    public T Value { get; }
    public Node<T>? Next { get; set; }
    public Node(T value) => Value = value;
}

Found 15 after 5 node visits.


The output is `Found 15 after 5 node visits.` The search cost matches the array scan exactly, because both inspect values in order. What has no counterpart here is `values[2]`: to reach the third value in the chain the program must follow two references from the head, so a "read by index" is itself a traversal. The costs table in the previous section records this as O(n) for indexed reads.


```{index} Middle Insertion Costs
```



```{index} Middle Insertion Costs
```

## Middle Insertion Costs



In an array, inserting into the middle requires shifting later elements. In a linked list, insertion after a known node changes references locally, but finding that node may still require traversal.

The array has four stored values and room for six. Inserting `30` at index 2 means every value from index 2 onward moves one slot to the right, starting from the end so that no value is overwritten before it is copied:

```csharp
for (int i = count; i > insertIndex; i--)
    values[i] = values[i - 1];     // shift right, last element first
values[insertIndex] = newValue;
count++;
```


In [ ]:
int[] values = new int[6] { 10, 20, 40, 50, 0, 0 };
int count = 4;
int insertIndex = 2;
int shifts = 0;

for (int i = count; i > insertIndex; i--)
{
    values[i] = values[i - 1];
    shifts++;
}

values[insertIndex] = 30;
count++;

Console.WriteLine(string.Join(", ", values[..count]));
Console.WriteLine($"shifts={shifts}");


10, 20, 30, 40, 50
shifts=2


The output is `10, 20, 30, 40, 50` with `shifts=2`: the values `40` and `50` each moved one slot. In general an insertion at index `k` shifts `count - k` values, so inserting at the front of a long array is expensive and appending at the end costs nothing. Shifting in the other direction, from `insertIndex` upward, is the classic mistake: it copies `40` over `50` before `50` has been moved, and the array ends up as `10, 20, 30, 40, 40`.


```{index} Guided Practice: Count the Deletion Shifts
```

### Guided Practice: Count the Deletion Shifts


Remove the value at index 1 from an array holding five values by shifting the later values left, and count the shifts. Expected output: `removed 20`, then `10, 30, 40, 50`, then `shifts=3`.


In [ ]:
using System;

int[] values = { 10, 20, 30, 40, 50 };
int count = 5;
int removeIndex = 1;
int shifts = 0;

int removed = values[removeIndex];
// Your code starts here.
// Shift every later value one slot to the left, counting each move.
// Your code ends here.
count--;

Console.WriteLine($"removed {removed}");
Console.WriteLine(string.Join(", ", values[..count]));
Console.WriteLine($"shifts={shifts}");


In [ ]:
using System;

// Solution
int[] values = { 10, 20, 30, 40, 50 };
int count = 5;
int removeIndex = 1;
int shifts = 0;

int removed = values[removeIndex];
for (int i = removeIndex; i < count - 1; i++)
{
    values[i] = values[i + 1];     // shift left, first element first
    shifts++;
}
count--;

Console.WriteLine($"removed {removed}");
Console.WriteLine(string.Join(", ", values[..count]));
Console.WriteLine($"shifts={shifts}");


removed 20
10, 30, 40, 50
shifts=3


Deletion shifts left and starts at the gap, the mirror image of insertion, which shifts right and starts at the end. Both move `count - index` values, so the costs table's O(n) for middle updates applies to each.


```{index} workload; operation count; representation comparison
```



```{index} Separate Location Cost from Update Cost
```

## Separate Location Cost from Update Cost



A service desk stores `n` work orders. Two requests that sound similar can require different work: “insert at position 400” supplies an index; “insert after this node” supplies a location handle. Compare the same request and the same assumptions before claiming a speed advantage.

| Request | Dynamic array | Singly linked list with head and tail |
| --- | --- | --- |
| Read index i | O(1) | O(i + 1), O(n) worst case |
| Append | O(1) amortized, O(n) on resize | O(1) with stored tail |
| Insert at front | O(n) shifts | O(1) |
| Insert by index in middle | O(n) shifts, possibly growth | O(n) locate, O(1) rewire |
| Remove by index in middle | O(n) shifts | O(n) locate, O(1) rewire |
| Remove last | O(1), without shrinking | O(n) find predecessor |
| Search unsorted values | O(n) | O(n) |
| Visit every item once | O(n) | O(n) with cursor |

The linked wrapper in this chapter exposes indices rather than public node handles. Its middle insertion therefore includes traversal. A different API could expose validated handles, but it must specify ownership and what happens when a handle refers to a removed node.

**Reason (applied):** A developer claims `Insert(400, value)` is constant-time because it changes two links. Identify the omitted work. Would adding a tail field fix it?

```{index} operation count; workload model
```



```{index} Count a Concrete Workload
```

## Count a Concrete Workload



Use eight initial orders and spare array capacity. Compare a front insertion, a last-position read, and a front removal.

| Operation | Array work | Linked work |
| --- | --- | --- |
| Insert at front | Eight existing-value shifts | No link traversal; update head |
| Read last position after insertion | One indexed read | Eight link advances from head |
| Remove front | Eight existing-value shifts | No link traversal; update head |

If readers request the last position 100 times between these updates, the array uses 100 indexed reads while the linked sequence uses 800 link advances. If the workload instead performs many front updates and few index reads, the preference may reverse. Include the whole operation mix.

Counts of different primitives are not interchangeable seconds. Copying a reference, following a link, allocating a node, and checking a bound have different costs. Counts explain how work grows; benchmarks answer a narrower question about a specific implementation and machine.

**Transfer (challenge):** Compare 100 last-index reads plus one front insertion for 1,000 orders. State your assumptions about spare capacity and any stored tail. Then change the request to “read the tail value.” Explain why changing the API changes the comparison.

```{index} boundary tests; invariants; sequence contract
```



```{index} Test the Contract Before Comparing Speed
```

## Test the Contract Before Comparing Speed



Both implementations should produce the same logical result for the same legal operation sequence. Test correctness first. A fast implementation that loses an order is not an alternative representation of the same ADT.

| Test | Operations | Expected result |
| --- | --- | --- |
| Empty start | Construct | Count zero; read index zero throws. |
| Single element | Add A; remove index zero | Returns A; count zero. |
| Reuse after emptying | Previous test; add B | Read index zero returns B. |
| Middle insertion | Add A, C; insert B at one | A, B, C in that order. |
| Middle removal | Previous test; remove one | Returns B; leaves A, C. |
| Duplicate values | Add A twice; remove zero | One A remains. |
| Invalid insertion | Insert at count plus one | Throws; count and contents unchanged. |
| Invalid removal | Remove at count | Throws; count and contents unchanged. |

Do not test only printed output from the happy path. Check count after each operation, and verify remaining values after removal. A useful additional check compares a generated operation sequence against C# `List<T>` as a reference implementation. It does not make timing comparisons fair; it checks observable behavior.

**Design (applied):** Write the smallest operation sequence that reveals a stale-tail bug. State the expected result before running it.

```{index} Worked Example: Check Both Implementations Against a Reference Model
```

### Worked Example: Check Both Implementations Against a Reference Model

The table above can be turned into code once and run against any implementation of the sequence contract. The example uses `List<T>` as the reference model and replays the same operation sequence on a candidate, comparing the observable results after every step: the count, the contents, and whether an invalid operation threw. The candidate here is the array-backed `WorkOrderArray<T>` from 14.1, reduced to the members the test needs; the linked `WorkOrderChain<T>` from 14.2 passes the same harness unchanged, because the harness talks only to the contract.

In [ ]:
using System;
using System.Collections.Generic;

var candidate = new WorkOrderArray<string>();
var model = new List<string>();
int failures = 0;

Step("add A", () => { candidate.Add("A"); model.Add("A"); });
Step("add C", () => { candidate.Add("C"); model.Add("C"); });
Step("insert B at 1", () => { candidate.Insert(1, "B"); model.Insert(1, "B"); });
Step("remove index 1", () => { candidate.RemoveAt(1); model.RemoveAt(1); });
Step("add A again", () => { candidate.Add("A"); model.Add("A"); });
Step("remove index 0", () => { candidate.RemoveAt(0); model.RemoveAt(0); });
ExpectThrow("insert at count + 1", () => candidate.Insert(candidate.Count + 1, "X"));
ExpectThrow("remove at count", () => candidate.RemoveAt(candidate.Count));
Step("state unchanged after invalid calls", () => { });
Console.WriteLine(failures == 0 ? "all checks passed" : $"{failures} check(s) failed");

void Step(string label, Action act)
{
    act();
    string got = $"[{string.Join(",", Snapshot())}] count={candidate.Count}";
    string want = $"[{string.Join(",", model)}] count={model.Count}";
    bool ok = got == want;
    if (!ok) failures++;
    Console.WriteLine($"{(ok ? "ok  " : "FAIL")} {label,-36} {got}");
}

void ExpectThrow(string label, Action act)
{
    try { act(); failures++; Console.WriteLine($"FAIL {label,-36} no exception"); }
    catch (ArgumentOutOfRangeException) { Console.WriteLine($"ok   {label,-36} threw as specified"); }
}

IEnumerable<string> Snapshot()
{
    for (int i = 0; i < candidate.Count; i++) yield return candidate[i];
}

public class WorkOrderArray<T>
{
    private T[] items = new T[2];
    public int Count { get; private set; }
    public T this[int index] { get { Check(index); return items[index]; } }
    private void Check(int index) { if (index < 0 || index >= Count) throw new ArgumentOutOfRangeException(nameof(index)); }
    public void Add(T value) => Insert(Count, value);
    public void Insert(int index, T value)
    {
        if (index < 0 || index > Count) throw new ArgumentOutOfRangeException(nameof(index));
        if (Count == items.Length) { var larger = new T[items.Length * 2]; Array.Copy(items, larger, Count); items = larger; }
        for (int i = Count; i > index; i--) items[i] = items[i - 1];
        items[index] = value;
        Count++;
    }
    public T RemoveAt(int index)
    {
        Check(index);
        T removed = items[index];
        for (int i = index; i < Count - 1; i++) items[i] = items[i + 1];
        Count--;
        items[Count] = default!;
        return removed;
    }
}

ok   add A                                [A] count=1
ok   add C                                [A,C] count=2
ok   insert B at 1                        [A,B,C] count=3
ok   remove index 1                       [A,C] count=2
ok   add A again                          [A,C,A] count=3
ok   remove index 0                       [C,A] count=2
ok   insert at count + 1                  threw as specified
ok   remove at count                      threw as specified
ok   state unchanged after invalid calls  [C,A] count=2
all checks passed


Every line reports `ok`, and the final line says all checks passed. The value of the harness is in what it would print for a defective candidate: a `RemoveAt` that forgot to decrement `Count` fails at the first removal with `count=3` against the model's `count=2`, and an `Insert` that shifted in the wrong direction fails with `[A,B,B]` against `[A,B,C]`. The two `ExpectThrow` checks confirm the failure policy, and the step after them confirms that a rejected call left the contents alone, which a candidate that incremented `Count` before validating would fail. Swap the construction line to `new WorkOrderChain<string>()` from 14.2 and the same program checks the linked representation; that is what it means for two data structures to implement one ADT.

```{index} memory locality; reference type; allocation
```



```{index} Memory and Locality Qualify the Recommendation
```

## Memory and Locality Qualify the Recommendation



For `int` elements, the backing array places the integer values together. For reference-type elements, the reference slots are contiguous but their target objects need not be. Sequential array access often benefits from memory locality. A linked traversal follows separately allocated node objects, which may be scattered in memory.

Linked lists also pay for a node object and link field per item. Dynamic arrays pay for spare capacity and occasional replacement arrays. Neither representation has universally smaller memory usage: the element type, spare capacity, object layout, and workload matter.

Begin with a dynamic array such as `List<T>` for ordinary indexed application sequences. Justify a linked structure when its operations and API offer a concrete benefit, such as repeated front removal or insertion at already known positions. Do not replace every `List<T>` just because some linked updates have constant cost.

**Decision memo (challenge):** Recommend a representation for a dispatch application. Give two frequent operations, one worst-case risk, a memory tradeoff, and a measurement that could change your recommendation.

```{index} Decision Guide
```



```{index} Decision Guide
```

## Decision Guide



| Workload | Better starting representation | Why |
| --- | --- | --- |
| Frequent indexing | Dynamic array | O(1) direct access by index |
| Frequent append | Dynamic array | Amortized O(1) append |
| Frequent insertion after known nodes | Linked list | O(1) reference update |
| Frequent search by value | Neither is ideal alone | Both may scan O(n) |
| Low per-item memory overhead | Dynamic array | No per-node reference field |

Use this table as a starting point, then adjust based on real operation counts and program constraints.

Two additions make the guide usable in practice. First, the C# library already provides both representations: `List<T>` is the dynamic array and `LinkedList<T>` is a doubly linked list with public node handles, so "choose a representation" usually means choosing between these two classes rather than writing one. Second, the guide's rows interact. A playlist that is mostly skipped by index but occasionally reordered is an indexing workload with a few O(n) shifts, which `List<T>` handles well; a simulation that constantly inserts events next to a known event and never asks for position 500 is the rare case where the linked class earns its memory. When no row clearly dominates, start with `List<T>`, measure the real operation mix, and switch only on evidence.

```{index} Cost Ordering: Rank the Operations
```

### Cost Ordering: Rank the Operations

For a sequence of 10,000 work orders, rank these five operations from cheapest to most expensive, first for a dynamic array and then for a singly linked list with head and tail references. Assume the array has spare capacity and that no node reference is held in advance.

A. Read the item at index 9,999. B. Insert a new item at index 0. C. Append a new item. D. Remove the item at index 5,000. E. Remove the last item.

```{dropdown} Answer
Dynamic array, cheapest first: A and C tie (one read; one write), then E (one count decrement and a cleared slot), then D (about 5,000 shifts), then B (10,000 shifts).

Singly linked list, cheapest first: B and C tie (one relink at the head or tail), then D and E (about 5,000 and 9,999 link advances respectively to reach the predecessor), then A (9,999 link advances to read the last value; with a tail reference, "read the tail value" would be O(1), but "read index 9,999" as stated is a traversal).

The two rankings are nearly reversed, which is the whole lesson of this lab: the question is never which representation is faster, but faster at which operations.
```

```{index} Lab Tasks
```



```{index} Lab Tasks
```

## Lab Tasks



Apply the preceding workload and verification reasoning here. The first two tasks check the earlier traces; the remaining tasks require a method implementation and justified representation choices.


1. Change the target in both search examples to the first, middle, last, and missing values. Record visit counts.
2. Modify the array insertion example to insert at the front. How many shifts are required?
3. Write a linked-list `CountNodes` method and test it with empty, one-node, and three-node lists.
4. Choose a representation for a playlist app where users often skip by index.
5. Choose a representation for a simulation where events are frequently inserted after a known event node.


```{index} Check Your Understanding
```

### Check Your Understanding

Answer each question before opening its dropdown.

1. A benchmark shows the linked list inserting at the front 400 times faster than the array. A teammate proposes switching the whole application to linked lists. What question do you ask first?

```{dropdown} Answer
What fraction of the real workload is front insertion? If the application mostly reads by index or appends, the array wins those operations by a similar margin, and the overall effect of switching could be negative. Count the whole operation mix before choosing.
```

2. Both implementations pass the reference-model harness. Does that make them equally correct, and does it make them equally fast?

```{dropdown} Answer
Equally correct for the operation sequences tested, yes: the harness compares observable behavior against a trusted model. It says nothing about speed, and a harness that passes is the precondition for a fair timing comparison, not a substitute for one.
```

3. Why does the comparison table list "insert by index in the middle" as O(n) for the linked list when the relinking itself is two assignments?

```{dropdown} Answer
Because the index must be turned into a node reference first, and that is a traversal from the head. The table charges each request for all the work it causes, not just for its last step.
```

4. The array stores 8-byte references to work-order objects; the linked list stores the same references plus a next link. Which uses less memory for 1,000 orders in an array allocated for 1,024?

```{dropdown} Answer
The array: about 1,024 × 8 = 8,192 bytes of slots against about 1,000 × 16 = 16,000 bytes of nodes, before counting the node objects' own headers. The array's waste is 24 unused slots; the list's overhead is one link per item. At low occupancy the comparison would reverse, as the break-even formula in 13.2 showed.
```

```{index} summary
```



```{index} Summary
```

## Summary



Arrays and linked lists both represent sequences, but they optimize different operations. A strong data-structure choice starts with the workload: what the program does most often, what must be fast, and what tradeoffs are acceptable.


```{rubric} Footnotes
```


[^1403_comparison_lab-1]: Later chapters revisit these same tradeoffs in stacks, queues, trees, hash tables, and graphs.
